# Multi-view images → 3D mesh (free GPU)

Runs an open-source image-to-3D model on Colab's free T4. Upload the views you
prepped with `prep_views.py`, get a mesh back.

**Before you run this, read `pipeline/README.md`** — in particular the licensing
section. You are planning to sell the output, and the model's licence governs
whether you can.

**Runtime → Change runtime type → T4 GPU** before running anything.

> These research repos move quickly. If the install cell fails, check the
> project's current README — the clone URL and entry point are the parts most
> likely to have changed since this notebook was written.

## 1. Confirm you actually have a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
print('GPU ready:', torch.cuda.get_device_name(0))

## 2. Install

Takes several minutes. The model weights are a few GB and are cached to the
session — a disconnect means downloading them again.

In [ ]:
%cd /content
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git hunyuan3d 2>/dev/null || echo 'already cloned'
%cd /content/hunyuan3d
!pip install -q -r requirements.txt
!pip install -q -e .
print('\ninstall step finished — check above for errors')

## 3. Upload your prepped views

The four PNGs written by `prep_views.py` — `front.png`, `back.png`,
`left.png`, `right.png`. Backgrounds must already be transparent.

In [ ]:
import os
from google.colab import files

os.makedirs('/content/views', exist_ok=True)
for name, data in files.upload().items():
    with open(f'/content/views/{name}', 'wb') as fh:
        fh.write(data)

print('\nuploaded:', sorted(os.listdir('/content/views')))

## 4. Generate the mesh

`octree_resolution` is the geometry-detail dial and the main thing to raise if
the result looks soft. Higher costs VRAM — drop it if the T4 runs out.

In [ ]:
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

# The multi-view checkpoint ('mv') conditions on several views at once, which
# is the whole point of shooting front/back/left/right. A single-view model
# would throw away three of your four references.
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mv'
)

views = {}
for key in ['front', 'back', 'left', 'right']:
    path = f'/content/views/{key}.png'
    if os.path.exists(path):
        views[key] = Image.open(path).convert('RGBA')
print('conditioning on:', sorted(views))

mesh = pipeline(
    image=views,
    num_inference_steps=50,
    octree_resolution=380,
    guidance_scale=5.0,
)[0]

mesh.export('/content/generated.glb')
print('\nwrote /content/generated.glb')

## 5. Download

Then run it through `finish_mesh.py` locally to fix scale, origin, smoothing
and LODs before it goes anywhere near a buyer.

In [ ]:
from google.colab import files
files.download('/content/generated.glb')